# Phase A2: SAM3 + LoRA Fine-tune on PanNuke (TRAIN ONLY)

**Goal:** Match SAM3-Adapter baseline (Kong et al. 2025 Fig 5, ~30-40% macro mIoU).

**Setup:**
- Backbone SAM3 frozen, LoRA rank=16 on decoder attention (~5-8M trainable params)
- **Train**: PanNuke Fold 1 + Fold 2 ONLY (~5179 patches)
- **Val/Eval**: SKIP here — Fold 3 strictly held out for `sam3_pannuke_phaseA2_eval.ipynb`
- Gamper protocol: no Fold 3 leak into training

**Prerequisites Kaggle:**
1. GPU: T4 x2 hoặc P100
2. Internet: ON
3. Datasets:
   - `hipinhththu/pannuke`
   - `hipinhththu/sam3-native-pt`

**Compute budget (2 epoch, LR 3e-4, full Fold1+2):**
- Kaggle T4: ~9-10h (margin 2-3h trong 12h session)
- Colab A100: ~3-4h

*Note:* config đã tối ưu cho Kaggle T4. Trên A100 có thể tăng `NUM_EPOCHS=3` và
hạ `LEARNING_RATE=2e-4` để gain thêm ~2-3pp mIoU.

## 00 — Setup

In [ ]:
import subprocess, sys, os, platform, time, json
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
import torch
print("Torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

In [ ]:
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"
DATA_ROOT = "/kaggle/input/datasets/hipinhththu/pannuke"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

assert os.path.exists(CHECKPOINT_PATH), "Attach hipinhththu/sam3-native-pt"
assert os.path.exists(DATA_ROOT), "Attach hipinhththu/pannuke"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scikit-learn", "matplotlib", "opencv-python",
                "pycocotools", "einops", "tqdm"], check=True)
print("OK setup")

## Helper modules (writefile)

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile lora_sam3.py
from __future__ import annotations
import math
from typing import Iterable, List, Optional, Set, Tuple
import torch
import torch.nn as nn

class LoRALinear(nn.Module):

    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32,
                 dropout: float = 0.05):
        super().__init__()
        self.base = base
        
        for p in self.base.parameters():
            p.requires_grad = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_f = base.in_features
        out_f = base.out_features
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return out + lora_out * self.scaling

    @property
    def in_features(self) -> int:
        return self.base.in_features

    @property
    def out_features(self) -> int:
        return self.base.out_features

DEFAULT_LORA_TARGETS: Set[str] = {
    
    
    
    "linear1", "linear2",
}

def inject_lora(
    model: nn.Module,
    target_module_names: Iterable[str] = DEFAULT_LORA_TARGETS,
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
    path_must_contain: str = "decoder",
    verbose: bool = True,
) -> Tuple[List[str], int]:
    target_set = set(target_module_names)
    replaced: List[str] = []

    for parent_name, parent in list(model.named_modules()):
        
        if path_must_contain and path_must_contain not in parent_name:
            continue
        for attr_name, child in list(parent.named_children()):
            if attr_name in target_set and isinstance(child, nn.Linear):
                lora_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
                
                base_device = next(child.parameters()).device
                lora_layer.lora_A.data = lora_layer.lora_A.data.to(base_device)
                lora_layer.lora_B.data = lora_layer.lora_B.data.to(base_device)
                setattr(parent, attr_name, lora_layer)
                full_path = f"{parent_name}.{attr_name}" if parent_name else attr_name
                replaced.append(full_path)

    n_lora_params = sum(p.numel() for p in model.parameters()
                        if p.requires_grad)

    if verbose:
        print(f"Injected LoRA into {len(replaced)} modules.")
        print(f"LoRA trainable params: {n_lora_params:,} "
              f"({n_lora_params/1e6:.2f}M)")
        if len(replaced) > 0:
            print("Sample paths:")
            for p in replaced[:5]:
                print(f"  {p}")
            if len(replaced) > 5:
                print(f"  ... ({len(replaced)-5} more)")

    return replaced, n_lora_params

def freeze_non_lora(model: nn.Module) -> Tuple[int, int]:
    n_train = 0
    n_total = 0
    for name, p in model.named_parameters():
        n_total += p.numel()
        if "lora_A" in name or "lora_B" in name:
            p.requires_grad = True
            n_train += p.numel()
        else:
            p.requires_grad = False
    return n_train, n_total

def save_lora_state(model: nn.Module, path: str):
    state = {n: p.detach().cpu()
             for n, p in model.named_parameters()
             if ("lora_A" in n or "lora_B" in n)}
    torch.save(state, path)
    return len(state)

def load_lora_state(model: nn.Module, path: str) -> int:
    state = torch.load(path, map_location="cpu")
    own = dict(model.named_parameters())
    n_loaded = 0
    for k, v in state.items():
        if k in own:
            own[k].data = v.to(own[k].device)
            n_loaded += 1
    return n_loaded

In [ ]:
%%writefile sam3_train.py
from __future__ import annotations
from typing import Dict, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

def make_transform(resolution: int = 1008):
    return v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(resolution, resolution)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

@torch.no_grad()
def encode_image_frozen(model, transform, pil_img, device: str = "cuda"):
    image = v2.functional.to_image(pil_img).to(device)
    image = transform(image).unsqueeze(0)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        backbone_out = model.backbone.forward_image(image)

        
        inst_pred = getattr(model, "inst_interactive_predictor", None)
        if inst_pred is not None and "sam2_backbone_out" in backbone_out:
            sam2_bb = backbone_out["sam2_backbone_out"]
            sam2_bb["backbone_fpn"][0] = (
                inst_pred.model.sam_mask_decoder.conv_s0(sam2_bb["backbone_fpn"][0])
            )
            sam2_bb["backbone_fpn"][1] = (
                inst_pred.model.sam_mask_decoder.conv_s1(sam2_bb["backbone_fpn"][1])
            )
    return backbone_out

def encode_text(model, prompt: str, device: str = "cuda"):
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            return model.backbone.forward_text([prompt], device=device)

def forward_decoder_with_grad(model, backbone_out: Dict, find_stage,
                              geometric_prompt, device: str = "cuda") -> Dict:
    was_training = model.training
    if was_training:
        model.eval()
    try:
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            outputs = model.forward_grounding(
                backbone_out=backbone_out,
                find_input=find_stage,
                geometric_prompt=geometric_prompt,
                find_target=None,
            )
    finally:
        if was_training:
            model.train()
    return outputs

def semantic_union_mask(outputs: Dict, target_hw: Tuple[int, int]) -> torch.Tensor:
    
    pred_logits = outputs["pred_logits"].float()         
    pred_masks  = outputs["pred_masks"].float()          
    pres_logit  = outputs["presence_logit_dec"].float()  

    
    cls_prob = pred_logits.sigmoid()                   
    pres     = pres_logit.sigmoid().unsqueeze(1)       
    prob     = (cls_prob * pres).squeeze(-1)           

    
    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid()  

    
    weighted = prob[:, :, None, None] * masks   

    
    
    
    eps = 1e-4
    log_one_minus = torch.log(torch.clamp(1.0 - weighted, min=eps, max=1.0 - eps))
    log_prod = log_one_minus.sum(dim=1)         
    union = 1.0 - torch.exp(torch.clamp(log_prod, min=-20.0))  
    union = torch.clamp(union, min=eps, max=1.0 - eps)  
    return union.squeeze(0)                     

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def bce_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    p = torch.clamp(pred, eps, 1 - eps)
    return -(target * torch.log(p) + (1 - target) * torch.log(1 - p)).mean()

def semantic_seg_loss(pred: torch.Tensor, target: torch.Tensor,
                      bce_weight: float = 0.5,
                      dice_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    total = bce_weight * bce + dice_weight * dice
    return total, {"bce": float(bce.item()), "dice": float(dice.item()),
                   "loss": float(total.item())}

@torch.no_grad()
def inference_to_binary(outputs: Dict, target_hw: Tuple[int, int],
                        score_threshold: float = 0.3) -> torch.Tensor:
    pred_logits = outputs["pred_logits"]
    pred_masks  = outputs["pred_masks"]
    pres_logit  = outputs["presence_logit_dec"]

    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)  

    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)  

    keep = prob > score_threshold
    if keep.sum() == 0:
        return torch.zeros(target_hw, dtype=torch.bool, device=pred_logits.device)

    selected = (masks[keep] > 0.5)
    union = selected.any(dim=0)
    return union

In [ ]:
import sys
for p in [".", "/kaggle/working", SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pannuke_loader import PanNukeFold, CELL_TYPES, DEFAULT_ROOT
from metrics import ClassWiseAccumulator, union_masks
from lora_sam3 import (LoRALinear, inject_lora, freeze_non_lora,
                       save_lora_state, load_lora_state, DEFAULT_LORA_TARGETS)
from sam3_train import (make_transform, encode_image_frozen, encode_text,
                        forward_decoder_with_grad, semantic_union_mask,
                        semantic_seg_loss, inference_to_binary)

print("Helpers loaded.")

## 01 — Build SAM3 + Inject LoRA

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Build SAM3 (fp32, frozen backbone)...")
model = build_sam3_image_model(
    device=device, eval_mode=True,
    checkpoint_path=CHECKPOINT_PATH, load_from_HF=False,
)
model.eval()
n_total = sum(p.numel() for p in model.parameters())
print(f"SAM3 params: {n_total/1e6:.1f}M")

In [ ]:
print("Decoder Linear submodules (verify LoRA targets):")
count = 0
for name, mod in model.named_modules():
    if isinstance(mod, torch.nn.Linear) and "decoder" in name.lower():
        attr_name = name.split(".")[-1]
        in_lora_targets = attr_name in DEFAULT_LORA_TARGETS
        marker = " <- LoRA target" if in_lora_targets else ""
        print(f"  {name}: {mod.in_features}->{mod.out_features}{marker}")
        count += 1
        if count >= 30:
            print(f"  ... (showing first 30 of many)")
            break
print(f"\nDefault LoRA targets: {sorted(DEFAULT_LORA_TARGETS)}")

In [ ]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

replaced, n_lora = inject_lora(
    model,
    target_module_names=DEFAULT_LORA_TARGETS,
    r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT,
)

n_train, n_total = freeze_non_lora(model)
print(f"\nTrainable: {n_train/1e6:.2f}M / Total: {n_total/1e6:.1f}M  "
      f"({100*n_train/n_total:.3f}%)")

if len(replaced) == 0:
    print("WARN: KHONG LoRA module nao duoc inject. Check decoder.py structure.")
    print("      Possible fix: extend DEFAULT_LORA_TARGETS trong lora_sam3.py")

## 02 — Dataloader (PanNuke Fold 1+2 train ONLY, no Fold 3 here)

In [ ]:
fold1 = PanNukeFold(DEFAULT_ROOT, 1)
fold2 = PanNukeFold(DEFAULT_ROOT, 2)

train_size = len(fold1) + len(fold2)
print(f"Train: Fold 1+2 = {train_size} patches")
print("Fold 3 NOT loaded here — strictly held out for eval (Gamper protocol).")

PROMPTS_LLM = {
    "Neoplastic":   ["Neoplastic cell", "Tumor cell", "Cancer cell", "Malignant cell"],
    "Inflammatory": ["Inflammatory cell", "Lymphocyte", "Immune cell", "Leukocyte"],
    "Connective":   ["Connective tissue cell", "Fibroblast", "Stromal cell"],
    "Dead":         ["Dead cell", "Apoptotic cell", "Necrotic cell"],
    "Epithelial":   ["Epithelial cell", "Epithelium", "Lining cell",
                     "Surface cell", "Mucosal cell nucleus"],
}

In [ ]:
import random
import numpy as np
from PIL import Image

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

class PanNukeTrainStream:
    """Stream (image, class_idx, gt_mask, prompt) samples cho training.

    Sample (img_i, class_k) random, lay random synonym lam prompt.
    """
    def __init__(self, folds, prompts_per_class, shuffle=True):
        self.folds = folds
        self.prompts = prompts_per_class
        self.cls_names = list(prompts_per_class.keys())
        self.index = []
        for fi, f in enumerate(folds):
            for i in range(len(f)):
                self.index.append((fi, i))
        if shuffle:
            random.shuffle(self.index)

    def __len__(self):
        return len(self.index) * len(self.cls_names)

    def iter_epoch(self):
        for fi, ii in self.index:
            sample = self.folds[fi][ii]
            img = Image.fromarray(sample["image"]).convert("RGB")
            for ck, cname in enumerate(self.cls_names):
                gt_mask = (sample["masks"][ck] > 0)
                if gt_mask.sum() < 10:
                    continue
                prompt = random.choice(self.prompts[cname])
                yield {
                    "image": img,
                    "class_name": cname,
                    "class_idx": ck,
                    "gt_mask": gt_mask,
                    "prompt": prompt,
                }

train_stream = PanNukeTrainStream([fold1, fold2], PROMPTS_LLM, shuffle=True)
print(f"Train stream: ~{len(train_stream)} (img,class) pairs/epoch (Fold 1+2 only)")
print("NOTE: NO val_stream — eval runs on separate eval notebook on Fold 3.")

## 03 — Training step (train-only, no eval here)

In [ ]:
from sam3.model.data_misc import FindStage

transform = make_transform(resolution=1008)

find_stage = FindStage(
    img_ids=torch.tensor([0], device=device, dtype=torch.long),
    text_ids=torch.tensor([0], device=device, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

def train_step(sample, optimizer):
    """1 (image, class) sample. Backprop qua LoRA params."""
    pil_img = sample["image"]
    prompt  = sample["prompt"]
    gt_mask = torch.from_numpy(sample["gt_mask"].astype(np.float32)).to(device)

    backbone_out = encode_image_frozen(model, transform, pil_img, device=device)
    text_out = encode_text(model, prompt, device=device)
    backbone_out.update(text_out)

    geometric_prompt = model._get_dummy_prompt()
    outputs = forward_decoder_with_grad(model, backbone_out,
                                         find_stage, geometric_prompt)

    pred = semantic_union_mask(outputs, target_hw=gt_mask.shape)

    loss, loss_dict = semantic_seg_loss(pred, gt_mask)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_norm=1.0
    )
    optimizer.step()

    return loss_dict

print("Training step ready. Eval is done in separate notebook (no val on Fold 3 here).")

## 04 — Training Loop (2 epochs, Kaggle-friendly, NO Fold 3 eval)

In [ ]:
from tqdm import tqdm

NUM_EPOCHS = 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
WARMUP_STEPS = 500
LOG_EVERY = 200
EVAL_EVERY = 1000
SAVE_EVERY = 500
MAX_TRAIN_PER_EPOCH = 5179

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS * MAX_TRAIN_PER_EPOCH
)

training_log = {
    "config": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "num_epochs": NUM_EPOCHS,
        "max_train_per_epoch": MAX_TRAIN_PER_EPOCH,
        "data_split": "Fold 1+2 train, NO Fold 3 val (eval on separate notebook)",
    },
    "epochs": [],
}

global_step = 0
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")
    model.train()
    epoch_losses = []

    pbar = tqdm(train_stream.iter_epoch(), total=MAX_TRAIN_PER_EPOCH,
                desc=f"Epoch {epoch+1}")
    for step, sample in enumerate(pbar):
        if step >= MAX_TRAIN_PER_EPOCH:
            break
        try:
            loss_dict = train_step(sample, optimizer)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print(f"OOM step {step}, skip")
                continue
            raise
        if not np.isfinite(loss_dict["loss"]):
            optimizer.zero_grad()
            if step < 50 or step % 200 == 0:
                print(f"  WARN: NaN/Inf loss at step {step}, skipped")
            continue
        if global_step < WARMUP_STEPS:
            warmup_lr = LEARNING_RATE * (global_step + 1) / WARMUP_STEPS
            for g in optimizer.param_groups:
                g["lr"] = warmup_lr
        else:
            scheduler.step()
        epoch_losses.append(loss_dict["loss"])
        global_step += 1

        if global_step % LOG_EVERY == 0:
            recent = epoch_losses[-LOG_EVERY:]
            pbar.set_postfix({"loss": f"{np.mean(recent):.4f}",
                              "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        if global_step % SAVE_EVERY == 0 and global_step > 0:
            from lora_sam3 import save_lora_state
            snap_path = f"{WORK}/lora_snapshot_step{global_step}.pt"
            save_lora_state(model, snap_path)
            import glob
            snaps = sorted(glob.glob(f"{WORK}/lora_snapshot_step*.pt"))
            for old in snaps[:-2]:
                os.remove(old)

    print(f"  Epoch {epoch+1} done. Avg loss: {np.mean(epoch_losses):.4f}")
    elapsed_epoch = time.time() - t0
    print(f"  Elapsed total: {elapsed_epoch/60:.1f} min")

    training_log["epochs"].append({
        "epoch": epoch + 1,
        "avg_loss": float(np.mean(epoch_losses)),
        "elapsed_seconds": elapsed_epoch,
        "note": "eval skipped — run sam3_pannuke_phaseA2_eval.ipynb on Fold 3 after training",
    })

    ckpt_path = f"{WORK}/sam3_lora_rank{LORA_R}_epoch{epoch+1}.pt"
    n_saved = save_lora_state(model, ckpt_path)
    print(f"  Saved LoRA state ({n_saved} tensors): {ckpt_path}")

print(f"\n===== Training done. Total: {(time.time()-t0)/60:.1f} min =====")

## 05 — Save Training Artifacts (no eval here)

**Eval đã tách sang notebook riêng** `sam3_pannuke_phaseA2_eval.ipynb` (Fold 3 holdout).

Notebook này CHỈ training + save LoRA weights. Fold 3 chưa bao giờ được load trong notebook này.

In [ ]:
import json

final_ckpt = f"{WORK}/sam3_lora_rank{LORA_R}_final.pt"
save_lora_state(model, final_ckpt)
print(f"Saved final LoRA: {final_ckpt}")

training_out = {
    "config": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lora_targets": sorted(DEFAULT_LORA_TARGETS),
        "n_modules": len(replaced),
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "max_train_per_epoch": MAX_TRAIN_PER_EPOCH,
    },
    "training_log": training_log,
    "elapsed_total_minutes": (time.time() - t0) / 60,
}
with open(f"{WORK}/phase_A2_training_log.json", "w") as f:
    json.dump(training_out, f, indent=2)
print(f"Saved log: {WORK}/phase_A2_training_log.json")

import glob
saved = sorted(glob.glob(f"{WORK}/sam3_lora_*.pt"))
print("\nAll LoRA checkpoints saved:")
for p in saved:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {p}  ({size_mb:.2f} MB)")

print("\n" + "=" * 70)
print("PHASE A2 TRAINING DONE")
print("=" * 70)
print(f"  Final LoRA weights: {final_ckpt}")
print(f"  Training log     : {WORK}/phase_A2_training_log.json")
print()
print("  Next step:")
print("  1. Download {final_ckpt} (or save vao Kaggle Dataset)")
print("  2. Open notebook `sam3_pannuke_phaseA2_eval.ipynb`")
print("  3. Upload LoRA checkpoint + Run All -> 3 strategies x 5 classes eval")
print("=" * 70)

## Phase A2 PASS criteria

- **Train loss giảm đều** qua 2 epochs (no NaN)
- **Val mIoU tăng** so với pre-training (Phase A1 baseline)
- **Best strategy match SAM3-Adapter target**: macro mIoU 25-40%
  (Kong et al. 2025 Fig 5 reported ~30% on PanNuke for SAM3-Adapter)
- **Per-class IoU**: Neoplastic > 30%, các classes khác > 15%, Dead có thể thấp (rare)

**Nếu macro mIoU < 15% sau training:**
- Check LoRA injection (cell INSPECT in §01) — có đúng module names không
- Tăng LORA_R lên 32, hoặc tăng LR lên 5e-4
- Verify GT masks load đúng (visualize 1 sample)

**Outputs saved to /kaggle/working/:**
- `sam3_lora_rank16_final.pt` — LoRA weights (~10 MB)
- `sam3_lora_rank16_epoch{1,2}.pt` — checkpoints sau mỗi epoch
- `phase_A2_training_log.json` — loss curve + pre-train eval

**Next steps:**
- **Notebook eval riêng**: `sam3_pannuke_phaseA2_eval.ipynb` — full Fold 3 với 3 strategies
- Phase A3: Type head (Linear 256→5) on top of fine-tuned model
- Phase C: JCC main table (5 conformal methods × 3 settings)